Reproduction Experiment D1

Setup Google Drive and install necessary packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/ConceptualizingConceptDrift')
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/drive/MyDrive/ConceptualizingConceptDrift']


In [ ]:
!pip install xplique timm opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 157.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 74.3 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


Load and preprocess NINCO images.


In [ ]:
from datasets import load_dataset
from PIL import Image
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from concept_helpers.DeepView_Craft import CraftTorchDV as Craft
from concept_helpers.DeepView_Craft import CraftTorchSupervised as CraftS
from concept_helpers.combined_crafts import CombinedCrafts

import urllib.request
import glob
import torch
import torch.nn as nn
from torchvision import transforms
import timm

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from scipy.sparse.linalg import eigs
from sklearn.ensemble import  RandomForestClassifier

import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import NMF
from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

import random

from xplique.concepts.craft import BaseCraft, DisplayImportancesOrder, Factorization, Sensitivity
from sklearn.decomposition import non_negative_factorization
from experiment_helpers.helper_function import *
from experiment_helpers.driftLocalizer import Localizer

# loading pretrained ResNet, create Data Configuration Pipeline, convert Images to correct Inputs
device = 'cuda'
model = timm.create_model('nf_resnet50.ra2_in1k', pretrained=True)
model = model.to(device)
config = resolve_data_config({}, model=model)
transform = create_transform(**config)
to_pil = transforms.ToPILImage()

# cut the model in twop arts (as explained in the paper)
# first part is g(.) our 'input_to_latent' model, second part is h(.) our 'latent_to_logit' model
g = nn.Sequential(*(list(model.children())[:4]))  # input to penultimate layer
h = nn.Sequential(*(list(model.children())[4:]))  # penultimate layer to logits


with urllib.request.urlopen('https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt') as f:
        imagenet_class_names = np.array(f.read().decode('utf-8').split('\n'))

def gen_images(filelist,folder_names,folder_name2class_id):
        for f in filelist:
            folder_name = f.split('/')[-2]
            if folder_name in folder_names:
                class_id = folder_name2class_id[folder_name]
                im = Image.open(f)
                if len(im.getbands()) == 3:
                    yield np.array(im.resize((224, 224))), class_id

# ood_folder = 'data/ninco_data/NINCO/NINCO_OOD_classes'
ood_folder = '/content/drive/MyDrive/data/ninco/NINCO/NINCO_OOD_classes'

ood_folder_names = ['french_fries','donuts','waffles','glass_of_milk','cup_cakes','chicken_quesadilla']
ood_class_names = ['french_fries','donuts','waffles','glass_of_milk','cup_cakes','chicken_quesadilla']


ood_class_ids = [1001 + i for i,class_name in enumerate(ood_class_names)]
ood_folder_name2class_id = dict(zip(ood_folder_names, ood_class_ids))
ood_filelist = glob.glob(f'{ood_folder}/*/*.jpg')
# print(ood_filelist)

ood_images, ood_labels = zip(*gen_images(ood_filelist,ood_folder_names,ood_folder_name2class_id))
ood_images, ood_labels = np.array(ood_images), np.array(ood_labels)
ood_preprocessed_images = torch.stack([transform(to_pil(img)) for img in ood_images], 0)



model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

## Reproduction
50 runs of:
1. imitating abrupt drift with random drift labels per class
2. embedding full size images through foundation model
3. train drift localizer on embeddings and drift labels
4. fitting NMF per drift phase on patches to extract concepts into concept bank V_drift
5. representing activations according to basis V_drift
6. measure global, phase and local importances for concepts
7. running different h_tilde variations


In [ ]:
#create result arrays for every metric to report

label_maps = []
drift_ratios = []
drift_localizer = []
drift_comparison = []
drift_forest = []


one_local_one_global = []
one_local = []
two_local = []
three_local = []
one_global = []
two_global = []
three_global = []

one_local_one_global_preds = []
one_local_preds = []
two_local_preds = []
three_local_preds = []
one_global_preds = []
two_global_preds = []
three_global_preds = []

one_local_one_global_l = []
one_local_l = []
two_local_l = []
three_local_l = []
one_global_l = []
two_global_l = []
three_global_l = []

one_local_one_global_preds_l = []
one_local_preds_l = []
two_local_preds_l = []
three_local_preds_l = []
one_global_preds_l = []
two_global_preds_l = []
three_global_preds_l = []

# all_local_l = []
# all_global_l = []
# all_local_preds_l = []
# all_global_preds_l = []

one_local_l_probs = []
two_local_l_probs = []
three_local_l_probs = []
one_local_preds_l_probs = []
two_local_preds_l_probs = []
three_local_preds_l_probs = []

reconstructed_single_concepts = []
reconstructed_single_concepts_preds = []
reconstructed_2_concepts = []
reconstructed_2_concepts_preds = []
reconstructed_3_concepts = []
reconstructed_3_concepts_preds = []
reconstructed_all_concepts = []
reconstructed_all_concepts_preds = []



for j in range(50):

    sample_ids = np.random.choice(len(ood_preprocessed_images),500, False)
    sample_images = ood_preprocessed_images[sample_ids]
    # Initialize the label_map keys
    keys = ood_class_ids
    # Shuffle the keys for more randomness
    random.shuffle(keys)

    # Assign at least one of each label (0, 1, 2)
    initial_labels = [0, 1, 2]
    random.shuffle(initial_labels)

    # Ensure that the first three keys have 0, 1, and 2 respectively
    label_map = {keys[i]: initial_labels[i] for i in range(3)}

    # Randomly assign labels for the remaining keys
    for i in range(3, len(keys)):
        label_map[keys[i]] = random.randint(0, 2)

    label_maps.append(label_map)


    labels_mapped = np.array([label_map[class_id] for class_id in ood_labels])

    drift_labels = labels_mapped[sample_ids]

    # Randomly assign labels of 1 or 2 to samples with label 3
    #  (i.e., digits that occur both before and after the change point)
    label_2_idx = np.where(drift_labels == 2)[0]
    y_mixed = drift_labels.copy()
    y_mixed[label_2_idx] = np.random.choice([0, 1], size=len(label_2_idx))

    sample_labels = y_mixed

    drift_ratios.append({"BD": len(np.where(drift_labels == 0)[0]),
                         "AD": len(np.where(drift_labels == 1)[0]),
                         "Both": len(np.where(drift_labels == 2)[0])})

    full_size = 256
    patch_size= 100


    #Supervised CRAFT Training
    h_craftdv = CraftS(input_to_latent_model=g,
                        latent_to_logit_model=h,
                        number_of_concepts=5,
                        inputs=sample_images,
                        labels=sample_labels,
                        batch_size=64,
                        patch_size=full_size,
                        device=device)

    patches, patch_act, train_labels = h_craftdv._extract_patches(sample_images, sample_labels )

    bd_indices = np.where(sample_labels != 1)[0]
    ad_indices = np.where(sample_labels != 0)[0]

    bd_fit = Craft(input_to_latent_model=g,
                    latent_to_logit_model=h,
                    number_of_concepts=10,
                    # labels=h_y,
                    patch_size=patch_size,
                    batch_size=64,
                    device=device)
    print("Fitting Unsupervised Craft....")
    bd_crops, bd_crops_u, bd_w = bd_fit.fit(sample_images[bd_indices])


    ad_fit = Craft(input_to_latent_model=g,
                        latent_to_logit_model=h,
                        number_of_concepts=10,
                        # labels=h_y,
                        patch_size=patch_size,
                        batch_size=64,
                        device=device)
    print("Fitting Unsupervised Craft....")
    ad_crops, ad_crops_u, ad_w = ad_fit.fit(sample_images[ad_indices])

    drift_basis = np.vstack([bd_w, ad_w])

    drift_craft = CombinedCrafts(input_to_latent_model=g,
                    latent_to_logit_model=h,
                    number_of_concepts=len(drift_basis),
                    inputs=sample_images,
                    labels=sample_labels,
                    basis = drift_basis,
                    batch_size=64,
                    patch_size=patch_size,
                    device=device)
    print("Fitting Craft....")
    drift_craft.transform_all()


    X_clean = patch_act
    y_clean = train_labels

    # Initialize a random forest model with max_leaf_nodes=150
    localizer_model = Localizer()


    # Perform the train-test split on X_clean and sample_labels
    X_train_clean, X_test_clean, y_train, y_test = \
        train_test_split(X_clean, y_clean, train_size=0.7, random_state=42)

    # Fit the model to the mixed set (group 3 is randomly assigned to 1 or 2)
    print('Fitting Random Forest classifier...')
    localizer_model.fit(X_train_clean, y_train);
    print('Fitting complete.')

    localizer_bin_preds = localizer_model.l_predict(X_test_clean)
    drift_localizer.append(accuracy_score(localizer_bin_preds, y_test))

    drift_imp = np.round(estimate_importance_l(localizer_model, drift_craft, drift_basis, X_train_clean),3)



    # y_preds_l, _ = compute_predictions(localizer_model,X_test_clean)
    image_drift_imp_l = [estimate_importance_helper_l(drift_craft,localizer_model,drift_basis,
                                                  image,class_of_interest=localizer_bin_preds[i])
                               for i,image in enumerate(X_test_clean)]



    one_local_one_global_l.append(local_one_imp_concept_globally_l(drift_craft,image_drift_imp_l,y_test))
    one_local_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=1,labels=y_test))
    two_local_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=2,labels=y_test))
    three_local_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=3,labels=y_test))
    # all_local_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=20,labels=y_test))

    one_global_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=1,labels=y_test))
    two_global_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=2,labels=y_test))
    three_global_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=3,labels=y_test))
    # all_global_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=20,labels=y_test))

    # all_local_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=20,labels=y_test))


    one_local_one_global_preds_l.append(local_one_imp_concept_globally_l(drift_craft,image_drift_imp_l,localizer_bin_preds))
    one_local_preds_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=1,labels=localizer_bin_preds))
    two_local_preds_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=2,labels=localizer_bin_preds))
    three_local_preds_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=3,labels=localizer_bin_preds))


    # all_local_preds_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=20,labels=localizer_bin_preds))

    one_global_preds_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=1,labels=localizer_bin_preds))
    two_global_preds_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=2,labels=localizer_bin_preds))
    three_global_preds_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=3,labels=localizer_bin_preds))
    # all_global_preds_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=20,labels=localizer_bin_preds))


    localizer_bin_train_preds = localizer_model.l_predict(X_train_clean)
    image_drift_imp_l_train = [estimate_importance_helper_l(drift_craft,localizer_model,drift_basis,
                                                  image,class_of_interest=localizer_bin_train_preds[i])
                               for i,image in enumerate(X_train_clean)]
    concept_dist = concept_counter(image_drift_imp_l_train,localizer_bin_train_preds)

    one_local_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=1,labels=y_test))
    two_local_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=2,labels=y_test))
    three_local_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=3,labels=y_test))

    one_local_preds_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=1,labels=localizer_bin_preds))
    two_local_preds_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=2,labels=localizer_bin_preds))
    three_local_preds_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=3,labels=localizer_bin_preds))

    reconstructed_single_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=1)
    localizer_preds = localizer_model.l_predict(reconstructed_single_concept)
    reconstructed_single_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_single_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))


    reconstructed_2_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=2)
    localizer_preds = localizer_model.l_predict(reconstructed_2_concept)
    reconstructed_2_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_2_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))

    reconstructed_3_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=3)
    localizer_preds = localizer_model.l_predict(reconstructed_3_concept)
    reconstructed_3_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_3_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))


    reconstructed_all_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=21)
    localizer_preds = localizer_model.l_predict(reconstructed_all_concept)
    reconstructed_all_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_all_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))

    print("Run:",j+1)





In [ ]:
import csv


methods = [ drift_localizer,
            one_local_one_global_l,
            one_local_l,
            two_local_l,
            three_local_l,
            one_local_l_probs,
            two_local_l_probs,
            three_local_l_probs,

            one_global_l,
            two_global_l,
            three_global_l,

            one_local_one_global_preds_l,
            one_local_preds_l,
            two_local_preds_l,
            three_local_preds_l,
            one_local_preds_l_probs,
            two_local_preds_l_probs,
            three_local_preds_l_probs,

            one_global_preds_l,
            two_global_preds_l,
            three_global_preds_l,
            reconstructed_single_concepts,
            reconstructed_single_concepts_preds,
            reconstructed_2_concepts,
            reconstructed_2_concepts_preds,
            reconstructed_3_concepts,
            reconstructed_3_concepts_preds,
            reconstructed_all_concepts,
            reconstructed_all_concepts_preds,
            label_maps,
            drift_ratios]


method_names = ["drift_localizer",
                "one_local_one_global_l",
                "one_local_l",
                "two_local_l",
                "three_local_l",
                "one_local_l_probs",
                "two_local_l_probs",
                "three_local_l_probs",

                "one_global_l",
                "two_global_l",
                "three_global_l",

                "one _local_one_global_preds_l",
                "one_local_preds_l",
                "two_local_preds_l",
                "three_local_preds_l",
                "one_local_preds_l_probs",
                "two_local_preds_l_probs",
                "three_local_preds_l_probs",

                "one_global_preds_l",
                "two_global_preds_l",
                "three_global_preds_l",
                "reconstructed_single_concepts",
                "reconstructed_single_concepts_preds",
                "reconstructed_2_concepts",
                "reconstructed_2_concepts_preds",
                "reconstructed_3_concepts",
                "reconstructed_3_concepts_preds",
                "reconstructed_all_concepts",
                "reconstructed_all_concepts_preds",
                "label_maps",
                "drift_ratios"]

# Write to CSV
with open('/content/drive/MyDrive/results/paper_experimentD1_reproduced.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Method'] + [f'Run_{i+1}' for i in range(50)])  # Header row
    for method, accuracies in zip(method_names, methods):
        writer.writerow([method] + accuracies)


In [ ]:
import pandas as pd
import numpy as np

# Load CSV
# df = pd.read_csv('paper_experiment_D1.csv')
df = pd.read_csv('/content/drive/MyDrive/results/paper_experimentD1_repro.csv')
df = df.iloc[:29]

# Calculate mean and standard deviation
stats = {}
for method in df['Method']:
    accuracies = df[df['Method'] == method].drop('Method', axis=1).values.flatten().astype(float)
    mean = np.mean(accuracies)
    # median = np.median(accuracies)

    std = np.std(accuracies)
    # stats[method] = (mean, std, median)
    stats[method] = (mean, std)
# Example output for stats
print(stats)

{'drift_localizer': (np.float64(0.7986666666666667), np.float64(0.0798220242511774)), 'one_local_one_global_l': (np.float64(0.758), np.float64(0.05873102530463209)), 'one_local_l': (np.float64(0.7813333333333332), np.float64(0.055079740175292935)), 'two_local_l': (np.float64(0.7563333333333334), np.float64(0.04582454461190956)), 'three_local_l': (np.float64(0.7373333333333334), np.float64(0.05736433270479722)), 'one_local_l_probs': (np.float64(0.7756666666666667), np.float64(0.053718401564702825)), 'two_local_l_probs': (np.float64(0.7506666666666668), np.float64(0.04855237721608833)), 'three_local_l_probs': (np.float64(0.7333333333333334), np.float64(0.054853339815264565)), 'one_global_l': (np.float64(0.6763333333333332), np.float64(0.09943115988238072)), 'two_global_l': (np.float64(0.7243333333333333), np.float64(0.07233486941533339)), 'three_global_l': (np.float64(0.7449999999999999), np.float64(0.0681134674234431)), 'one_local_one_global_preds_l': (np.float64(0.7593333333333334), np

In [ ]:
latex_table = """
\\begin{table}[h!]
\\centering
\\begin{tabular}{l|c}
\\hline
Method & Accuracy (Mean ± Std Dev) \\\\
\\hline
"""

for method, (mean, std) in stats.items():
    latex_table += f"{method} & {mean:.3f} ± {std:.3f} \\\\ \n"

latex_table += """
\\hline
\\end{tabular}
\\caption{Accuracy of different methods}
\\end{table}
"""


#include all and see what happens

# Output the LaTeX table
print(latex_table)

##Model h tilde for paper is encompassed by "one_local_l_probs"

## We have other models here which use more concepts for possible future work